# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [11]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:98% !important; }</style>"))
%load_ext autoreload  
%autoreload 2
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [39]:
import re
import feedparser
from bs4 import BeautifulSoup
feed = feedparser.parse("https://www.dealnews.com/c142/Electronics/?rss=1")

In [48]:
def extract(html_snippet: str) -> str:
    """
    Use Beautiful Soup to clean up this HTML snippet and extract useful text
    """
    soup = BeautifulSoup(html_snippet, "html.parser")
    snippet_div = soup.find("div", class_="snippet summary")
    if snippet_div:
        print(f"Extracting from snippet  : {snippet_div}")
        description = snippet_div.get_text(strip=True)
        print(f"Extracted description    : {description}")
        description = BeautifulSoup(description, "html.parser").get_text()
        print(f"Description second pass  : {description}")
        description = re.sub("<[^<]+?>", "", description)
        print(f"Description regex cleanup: {description}")
        result = description.strip()
        print(f"Final extracted result   : {result}")
    else:
        result = html_snippet
    return result.replace("\n", " ")


In [51]:
entry = feed.entries[1]
entry

{'title': 'Amazon Renewed Phones: Get Deals on Apple, Samsung + free shipping',
 'title_detail': {'type': 'text/plain',
  'language': None,
  'base': 'https://www.dealnews.com/c142/Electronics/?rss=1',
  'value': 'Amazon Renewed Phones: Get Deals on Apple, Samsung + free shipping'},
 'links': [{'rel': 'alternate',
   'type': 'text/html',
   'href': 'https://www.dealnews.com/Amazon-Renewed-Phones-Get-Deals-on-Apple-Samsung-free-shipping/21811720.html?iref=rss-c142'}],
 'link': 'https://www.dealnews.com/Amazon-Renewed-Phones-Get-Deals-on-Apple-Samsung-free-shipping/21811720.html?iref=rss-c142',
 'summary': '<img src="https://www.datocms-assets.com/64599/1771616626-dark-gray-apple-smartphone-giws.webp?h=125&amp;w=125" style="float: left; vertical-align: top; margin: 0 8px 8px 0;" /><div class="snippet summary" title="In&#x20;this&#x20;section&#x20;you&#x27;ll&#x20;find&#x20;deals&#x20;on&#x20;a&#x20;large&#x20;selection&#x20;of&#x20;refurbished&#x20;and&#x20;preowned&#x20;phones&#x20;and&

In [52]:
entry['title']
entry["summary"]
extract(entry["summary"])

'Amazon Renewed Phones: Get Deals on Apple, Samsung + free shipping'

'<img src="https://www.datocms-assets.com/64599/1771616626-dark-gray-apple-smartphone-giws.webp?h=125&amp;w=125" style="float: left; vertical-align: top; margin: 0 8px 8px 0;" /><div class="snippet summary" title="In&#x20;this&#x20;section&#x20;you&#x27;ll&#x20;find&#x20;deals&#x20;on&#x20;a&#x20;large&#x20;selection&#x20;of&#x20;refurbished&#x20;and&#x20;preowned&#x20;phones&#x20;and&#x20;phone&#x20;accessories,&#x20;including&#x20;brands&#x20;like&#x20;Apple&#x20;and&#x20;Samsung."> <p>In this section you\'ll find deals on a large selection of refurbished and preowned phones and phone accessories, including brands like Apple and Samsung. Shop Now at Amazon </p> </div>'

Extracting from snippet  : <div class="snippet summary" title="In this section you'll find deals on a large selection of refurbished and preowned phones and phone accessories, including brands like Apple and Samsung."> <p>In this section you'll find deals on a large selection of refurbished and preowned phones and phone accessories, including brands like Apple and Samsung. Shop Now at Amazon </p> </div>
Extracted description    : In this section you'll find deals on a large selection of refurbished and preowned phones and phone accessories, including brands like Apple and Samsung. Shop Now at Amazon
Description second pass  : In this section you'll find deals on a large selection of refurbished and preowned phones and phone accessories, including brands like Apple and Samsung. Shop Now at Amazon
Description regex cleanup: In this section you'll find deals on a large selection of refurbished and preowned phones and phone accessories, including brands like Apple and Samsung. Shop Now at 

"In this section you'll find deals on a large selection of refurbished and preowned phones and phone accessories, including brands like Apple and Samsung. Shop Now at Amazon"

In [28]:
# type(entry)
# for key in entry.keys():
#     print(f" {key:20s}: {type(entry[key])}")
# print("\nFeed:")
# for key in entry['feed'].keys():
#     print(f" {key:20s}: {type(entry['feed'][key])}")
#     if isinstance(entry['feed'][key], feedparser.util.FeedParserDict):
#         for subkey in entry['feed'][key].keys():
#             print(f"\t{subkey:20s}: {type(entry['feed'][key][subkey])}")
#     elif isinstance(entry['feed'][key], str):
#         print(f"\t{entry['feed'][key][:100]}...")
# print("\nHeaders:")
# for key in entry['headers'].keys():
#     print(f" {key:20s}: {type(entry['headers'][key])}")

In [ ]:
        self.title = entry["title"]
        self.summary = extract(entry["summary"])
        self.url = entry["links"][0]["href"]
        stuff = requests.get(self.url).content
        soup = BeautifulSoup(stuff, "html.parser")
        content = soup.find("div", class_="content-section").get_text()
        content = content.replace("\nmore", "").replace("\n", " ")
        if "Features" in content:
            self.details, self.features = content.split("Features", 1)
        else:
            self.details = content
            self.features = ""
        self.truncate()

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

Fetching deals...


  0%|          | 0/3 [00:00<?, ?it/s]

*** Fetching https://www.dealnews.com/c142/Electronics/?rss=1
Processing Tech Savings at Walmart: Up to 71% off + free shipping w/ $35
	 Init ScrapedDeal
	 Entry: {'title': 'Tech Savings at Walmart: Up to 71% off + free shipping w/ $35', 'title_detail': {'type': 'text/plain', 'language': None, 'base': 'https://www.dealnews.com/c142/Electronics/?rss=1', 'value': 'Tech Savings at Walmart: Up to 71% off + free shipping w/ $35'}, 'links': [{'rel': 'alternate', 'type': 'text/html', 'href': 'https://www.dealnews.com/Tech-Savings-at-Walmart-Up-to-71-off-free-shipping-w-35/21811739.html?iref=rss-c142'}], 'link': 'https://www.dealnews.com/Tech-Savings-at-Walmart-Up-to-71-off-free-shipping-w-35/21811739.html?iref=rss-c142', 'summary': '<img src="https://www.datocms-assets.com/64599/1771620771-onn-soundbar-remote-nzn7.png?h=98&amp;w=125" style="float: left; vertical-align: top; margin: 0 8px 8px 0;" /><div class="snippet summary" title="Walmart&#x20;discounts&#x20;TVs,&#x20;laptops,&#x20;tablets,

 33%|███▎      | 1/3 [00:52<01:45, 52.57s/it]

*** Finished fetching https://www.dealnews.com/c142/Electronics/?rss=1
*** Fetching https://www.dealnews.com/c39/Computers/?rss=1
Processing Dell Refurbished Hot Deals: Up to 56% off + free shipping
	 Init ScrapedDeal
	 Entry: {'title': 'Dell Refurbished Hot Deals: Up to 56% off + free shipping', 'title_detail': {'type': 'text/plain', 'language': None, 'base': 'https://www.dealnews.com/c39/Computers/?rss=1', 'value': 'Dell Refurbished Hot Deals: Up to 56% off + free shipping'}, 'links': [{'rel': 'alternate', 'type': 'text/html', 'href': 'https://www.dealnews.com/Dell-Refurbished-Hot-Deals-Up-to-56-off-free-shipping/21811741.html?iref=rss-c39'}], 'link': 'https://www.dealnews.com/Dell-Refurbished-Hot-Deals-Up-to-56-off-free-shipping/21811741.html?iref=rss-c39', 'summary': '<img src="https://www.datocms-assets.com/64599/1771620924-2026-02-20_15-55-04.png?h=78&amp;w=125" style="float: left; vertical-align: top; margin: 0 8px 8px 0;" /><div class="snippet summary" title="Bag&#x20;deals&#x2

 67%|██████▋   | 2/3 [01:46<00:53, 53.32s/it]

*** Finished fetching https://www.dealnews.com/c39/Computers/?rss=1
*** Fetching https://www.dealnews.com/f1912/Smart-Home/?rss=1
Processing Samsung Q-Series 11.1.4 ch. Wireless Dolby ATMOS Soundbar w/ Rear Speakers for $1,000 + free shipping
	 Init ScrapedDeal
	 Entry: {'title': 'Samsung Q-Series 11.1.4 ch. Wireless Dolby ATMOS Soundbar w/ Rear Speakers for $1,000 + free shipping', 'title_detail': {'type': 'text/plain', 'language': None, 'base': 'https://www.dealnews.com/f1912/Smart-Home/?rss=1', 'value': 'Samsung Q-Series 11.1.4 ch. Wireless Dolby ATMOS Soundbar w/ Rear Speakers for $1,000 + free shipping'}, 'links': [{'rel': 'alternate', 'type': 'text/html', 'href': 'https://www.dealnews.com/products/Samsung/Samsung-Q-Series-11-1-4-ch-Wireless-Dolby-ATMOS-Soundbar-w-Rear-Speakers/478089.html?iref=rss-f1912'}], 'link': 'https://www.dealnews.com/products/Samsung/Samsung-Q-Series-11-1-4-ch-Wireless-Dolby-ATMOS-Soundbar-w-Rear-Speakers/478089.html?iref=rss-f1912', 'summary': '<img src="

100%|██████████| 3/3 [02:38<00:00, 52.85s/it]

*** Finished fetching https://www.dealnews.com/f1912/Smart-Home/?rss=1


In [4]:
len(deals)

30

In [5]:
deals[10].describe()

"Title: Dell Refurbished Hot Deals: Up to 56% off + free shipping\nDetails: Bag deals on this selection of refurbished Dell laptops, desktops, and workstations. We've pictured this zoom imageStock images may not represent exact product.Please read the product configuration and description details on this page. We've pictured the Refurbished Dell Latitude 5431 Touch Laptop for $449 ($370 off.) Plus, you'll get free shipping, and all systems come with a 100-day Dell warranty. Shop Now at Dell Refurbished Store\nFeatures: \nURL: https://www.dealnews.com/Dell-Refurbished-Hot-Deals-Up-to-56-off-free-shipping/21811741.html?iref=rss-c39"

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [53]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [54]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [57]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(f"Length of user prompt: {len(user_prompt)} characters\n")
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Length of user prompt: 15293 characters

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Tech Savings at Walmart: Up to 71% off + free shipping w/ $35
Details: Walmart discounts TVs, laptops, tablets, speakers, monitors, and a lot more. We've pictured the Okko 37" ClearWave TV Soundbar for $20 ($50 off). Get free shipping over $35, otherwise it adds $7. Shop Now at Walmart
Features: 
URL: https://www.dealnews.com/Tech-Saving

In [58]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Motorola Razr Ultra (2025) is a flagship foldable Android phone featuring a 7.0" Super HD main display paired with a 4.0" pOLED external display for quick interactions. It packs 16GB of RAM and a spacious 1TB internal storage, and a dual rear camera system led by a 50MP main shooter plus an ultrawide. The package includes Motorola\'s Moto Buds+ true wireless earbuds and supports modern hardware and multitasking demands in a clamshell design.', price=800.0, url='https://www.dealnews.com/products/Motorola/Motorola-Razr-Ultra-1-TB-Android-Phone-2025/495819.html?iref=rss-c142'), Deal(product_description='Samsung Q-Series 11.1.4-channel wireless Dolby Atmos soundbar includes a dedicated rear speaker kit to deliver immersive surround sound with object-based Dolby Atmos playback. The system integrates built-in Alexa and streaming smart services, plus SpaceFit and Adaptive Sound technologies to tune audio to your room. Its multi-channel configurat

In [59]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Motorola Razr Ultra (2025) is a flagship foldable Android phone featuring a 7.0" Super HD main display paired with a 4.0" pOLED external display for quick interactions. It packs 16GB of RAM and a spacious 1TB internal storage, and a dual rear camera system led by a 50MP main shooter plus an ultrawide. The package includes Motorola's Moto Buds+ true wireless earbuds and supports modern hardware and multitasking demands in a clamshell design.
800.0
https://www.dealnews.com/products/Motorola/Motorola-Razr-Ultra-1-TB-Android-Phone-2025/495819.html?iref=rss-c142

Samsung Q-Series 11.1.4-channel wireless Dolby Atmos soundbar includes a dedicated rear speaker kit to deliver immersive surround sound with object-based Dolby Atmos playback. The system integrates built-in Alexa and streaming smart services, plus SpaceFit and Adaptive Sound technologies to tune audio to your room. Its multi-channel configuration (11.1.4) targets home theater setups seeking high-end, room-filling audio without sepa

In [60]:
DealSelection.model_json_schema()

{'$defs': {'Deal': {'description': 'A class to Represent a Deal with a summary description',
   'properties': {'product_description': {'description': "Your clearly expressed summary of the product in 3-4 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a short paragraph of text for each item you choose.",
     'title': 'Product Description',
     'type': 'string'},
    'price': {'description': 'The actual price of this product, as advertised in the deal. Be sure to give the actual price; for example, if a deal is described as $100 off the usual $300 price, you should respond with $200',
     'title': 'Price',
     'type': 'number'},
    'url': {'description': 'The URL of the deal, as provided in the input',
     'title': 'Url',
     'type': 'string'}},
   'required': ['product_description', 'price', 'url'],
   'title': 'Deal',
   'type': 'object'}},
 'description': 'A clas

In [62]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [63]:
from agents.scanner_agent import ScannerAgent

In [64]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed


Fetching deals...
*** Fetching https://www.dealnews.com/c142/Electronics/?rss=1
Processing Tech Savings at Walmart: Up to 71% off + free shipping w/ $35
	 Init ScrapedDeal
	 Entry: {'title': 'Tech Savings at Walmart: Up to 71% off + free shipping w/ $35', 'title_detail': {'type': 'text/plain', 'language': None, 'base': 'https://www.dealnews.com/c142/Electronics/?rss=1', 'value': 'Tech Savings at Walmart: Up to 71% off + free shipping w/ $35'}, 'links': [{'rel': 'alternate', 'type': 'text/html', 'href': 'https://www.dealnews.com/Tech-Savings-at-Walmart-Up-to-71-off-free-shipping-w-35/21811739.html?iref=rss-c142'}], 'link': 'https://www.dealnews.com/Tech-Savings-at-Walmart-Up-to-71-off-free-shipping-w-35/21811739.html?iref=rss-c142', 'summary': '<img src="https://www.datocms-assets.com/64599/1771620771-onn-soundbar-remote-nzn7.png?h=98&amp;w=125" style="float: left; vertical-align: top; margin: 0 8px 8px 0;" /><div class="snippet summary" title="Walmart&#x20;discounts&#x20;TVs,&#x20;lapt

INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs


*** Finished fetching https://www.dealnews.com/f1912/Smart-Home/?rss=1


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [65]:
result

DealSelection(deals=[Deal(product_description='Motorola Razr Ultra (2025) is a foldable Android smartphone featuring a 7.0" 1224p Super HD main display and a 4.0" pOLED external display. It includes 16GB of RAM and a large 1TB of internal storage, plus a 50MP main rear camera with an ultrawide, and ships with a pair of Moto Buds+ wireless earbuds. The model is positioned as a high-capacity, flagship flip phone with modern performance and expandable multimedia capabilities.', price=800.0, url='https://www.dealnews.com/products/Motorola/Motorola-Razr-Ultra-1-TB-Android-Phone-2025/495819.html?iref=rss-c142'), Deal(product_description='Crucial T705 is a PCIe Gen5 NVMe M.2 solid-state drive with a 2TB capacity and a white heatsink. It delivers very high sequential performance with up to 14,500 MB/s reads and 12,700 MB/s writes, and strong random IOPS (up to 1,550K read / 1,800K write), making it suitable for high-end gaming rigs and workstation builds that need extreme throughput and low la

In [66]:
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed


Fetching deals...
*** Fetching https://www.dealnews.com/c142/Electronics/?rss=1
Processing Tech Savings at Walmart: Up to 71% off + free shipping w/ $35
	 Init ScrapedDeal
	 Entry: {'title': 'Tech Savings at Walmart: Up to 71% off + free shipping w/ $35', 'title_detail': {'type': 'text/plain', 'language': None, 'base': 'https://www.dealnews.com/c142/Electronics/?rss=1', 'value': 'Tech Savings at Walmart: Up to 71% off + free shipping w/ $35'}, 'links': [{'rel': 'alternate', 'type': 'text/html', 'href': 'https://www.dealnews.com/Tech-Savings-at-Walmart-Up-to-71-off-free-shipping-w-35/21811739.html?iref=rss-c142'}], 'link': 'https://www.dealnews.com/Tech-Savings-at-Walmart-Up-to-71-off-free-shipping-w-35/21811739.html?iref=rss-c142', 'summary': '<img src="https://www.datocms-assets.com/64599/1771620771-onn-soundbar-remote-nzn7.png?h=98&amp;w=125" style="float: left; vertical-align: top; margin: 0 8px 8px 0;" /><div class="snippet summary" title="Walmart&#x20;discounts&#x20;TVs,&#x20;lapt

INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs


*** Finished fetching https://www.dealnews.com/f1912/Smart-Home/?rss=1


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [67]:
load_dotenv(override=True)

True

In [68]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [69]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [70]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [76]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [78]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [77]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

AttributeError: 'ScannerAgent' object has no attribute 'notify'